# Análisis exploratorio del rendimiento académico de estudiantes

**Dataset:** `StudentsPerformance.csv`

## Objetivo
Analizar el desempeño académico de los estudiantes para identificar patrones relacionados con sus resultados en Matemáticas, Lectura y Escritura.

En este notebook se realizan:
- Carga y exploración inicial del dataset.
- Limpieza y preprocesamiento.
- Creación de la variable `average_score`.
- Clasificación del rendimiento académico.
- Más de 4 análisis sobre los datos.
- Más de 3 visualizaciones.
- Conclusiones finales.


## 1. Importar librerías y cargar el dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Cargar el dataset
df = pd.read_csv("StudentsPerformance.csv")

# Mostrar las primeras filas
df.head()


## 2. Exploración inicial

In [ ]:
# Número de registros y columnas
print("Número de registros:", df.shape[0])
print("Número de columnas:", df.shape[1])

print("\nNombre de las variables:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)


### Valores faltantes y registros duplicados

In [ ]:
# Valores faltantes por columna
print("Valores faltantes por columna:")
print(df.isnull().sum())

# Registros duplicados
print("\nNúmero de registros duplicados:", df.duplicated().sum())


### Estadísticas descriptivas

In [ ]:
# Estadísticas de las variables numéricas
df.describe()


## 3. Limpieza y preprocesamiento

Primero se revisa si existen valores faltantes o registros duplicados. En este dataset no se encontraron valores faltantes ni registros duplicados, por lo que no es necesario eliminar registros.

También se comprueba que las calificaciones estén dentro de un rango lógico de 0 a 100. Después se crean nombres de columnas más sencillos para facilitar el análisis.


In [ ]:
# Crear una copia para trabajar con los datos limpios
df_clean = df.copy()

# Cambiar nombres de columnas para trabajar más fácilmente
df_clean = df_clean.rename(columns={
    "race/ethnicity": "race_ethnicity",
    "parental level of education": "parental_education",
    "test preparation course": "test_preparation",
    "math score": "math_score",
    "reading score": "reading_score",
    "writing score": "writing_score"
})

# Comprobar rangos de las calificaciones
print("Rango de Matemáticas:", df_clean["math_score"].min(), "-", df_clean["math_score"].max())
print("Rango de Lectura:", df_clean["reading_score"].min(), "-", df_clean["reading_score"].max())
print("Rango de Escritura:", df_clean["writing_score"].min(), "-", df_clean["writing_score"].max())

df_clean.head()


## 4. Crear la variable `average_score`

La nueva variable representa el promedio de las tres áreas:

**Matemáticas + Lectura + Escritura / 3**


In [ ]:
df_clean["average_score"] = (
    df_clean["math_score"] +
    df_clean["reading_score"] +
    df_clean["writing_score"]
) / 3

df_clean[["math_score", "reading_score", "writing_score", "average_score"]].head()


## 5. Clasificación del rendimiento académico

Se utilizarán tres categorías:

- **Bajo:** promedio menor a 60.
- **Medio:** promedio de 60 a 79.99.
- **Alto:** promedio de 80 a 100.

Estos límites permiten separar a los estudiantes según su promedio general de las tres áreas.


In [ ]:
def clasificar_rendimiento(promedio):
    if promedio < 60:
        return "Bajo"
    elif promedio < 80:
        return "Medio"
    else:
        return "Alto"

df_clean["performance"] = df_clean["average_score"].apply(clasificar_rendimiento)

# Revisar algunos resultados
df_clean[["average_score", "performance"]].head(10)


### Distribución de las categorías de rendimiento

In [ ]:
performance_counts = df_clean["performance"].value_counts().reindex(["Bajo", "Medio", "Alto"])
performance_percent = (performance_counts / len(df_clean) * 100).round(2)

resultado_rendimiento = pd.DataFrame({
    "Cantidad": performance_counts,
    "Porcentaje": performance_percent
})

resultado_rendimiento


## 6. Análisis 1: ¿Cuál de las tres áreas tiene el promedio más alto?

In [ ]:
promedios_areas = df_clean[["math_score", "reading_score", "writing_score"]].mean().sort_values(ascending=False)

print("Promedio de cada área:")
print(promedios_areas.round(2))

area_mayor = promedios_areas.index[0]
print("\nEl área con el promedio más alto es:", area_mayor)


In [ ]:
# Visualización 1: promedio por área
plt.figure(figsize=(8, 5))
plt.bar(
    ["Matemáticas", "Lectura", "Escritura"],
    [
        df_clean["math_score"].mean(),
        df_clean["reading_score"].mean(),
        df_clean["writing_score"].mean()
    ]
)
plt.title("Promedio de calificaciones por área")
plt.ylabel("Promedio")
plt.ylim(0, 100)
plt.show()


## 7. Análisis 2: ¿Los estudiantes que realizaron el curso de preparación presentan mejores resultados?

Se compara el promedio general (`average_score`) entre los estudiantes que completaron el curso de preparación y quienes no lo realizaron.


In [ ]:
promedio_preparacion = (
    df_clean.groupby("test_preparation")["average_score"]
    .mean()
    .sort_values(ascending=False)
)

print(promedio_preparacion.round(2))

print("\nDiferencia de promedio:")
print(round(promedio_preparacion["completed"] - promedio_preparacion["none"], 2))


In [ ]:
# Visualización 2: promedio según curso de preparación
plt.figure(figsize=(8, 5))
plt.bar(
    ["Completó curso", "No realizó curso"],
    [
        promedio_preparacion["completed"],
        promedio_preparacion["none"]
    ]
)
plt.title("Promedio general según curso de preparación")
plt.ylabel("Promedio")
plt.ylim(0, 100)
plt.show()


## 8. Análisis 3: ¿Existen diferencias según el nivel educativo de los padres?

In [ ]:
promedio_educacion = (
    df_clean.groupby("parental_education")["average_score"]
    .mean()
    .sort_values(ascending=False)
)

print("Promedio general por nivel educativo de los padres:")
print(promedio_educacion.round(2))


In [ ]:
# Visualización 3: promedio según educación de los padres
plt.figure(figsize=(10, 6))
plt.barh(promedio_educacion.index, promedio_educacion.values)
plt.title("Promedio general según nivel educativo de los padres")
plt.xlabel("Promedio")
plt.xlim(0, 100)
plt.show()


## 9. Análisis 4: ¿Qué grupos presentan los promedios más altos y más bajos?

Se compara el promedio general entre los grupos de raza/etnia registrados en el dataset.


In [ ]:
promedio_grupo = (
    df_clean.groupby("race_ethnicity")["average_score"]
    .mean()
    .sort_values(ascending=False)
)

print("Promedio por grupo:")
print(promedio_grupo.round(2))

print("\nGrupo con mayor promedio:", promedio_grupo.index[0])
print("Grupo con menor promedio:", promedio_grupo.index[-1])


## 10. Análisis 5: ¿Qué porcentaje de estudiantes alcanza un promedio de 80 o más?

In [ ]:
porcentaje_80 = (df_clean["average_score"].ge(80).mean() * 100).round(2)
porcentaje_60 = (df_clean["average_score"].ge(60).mean() * 100).round(2)

print(f"Estudiantes con promedio de 80 o más: {porcentaje_80}%")
print(f"Estudiantes con promedio de 60 o más: {porcentaje_60}%")


In [ ]:
# Visualización 4: distribución de los promedios
plt.figure(figsize=(9, 5))
plt.hist(df_clean["average_score"], bins=15)
plt.axvline(60, linestyle="--", label="Límite Bajo/Medio (60)")
plt.axvline(80, linestyle="--", label="Límite Medio/Alto (80)")
plt.title("Distribución del promedio general")
plt.xlabel("Average score")
plt.ylabel("Número de estudiantes")
plt.legend()
plt.show()


## 11. Análisis adicional: ¿El tipo de almuerzo se relaciona con el rendimiento?

Se compara el promedio general de los estudiantes según el tipo de almuerzo registrado.


In [ ]:
promedio_lunch = (
    df_clean.groupby("lunch")["average_score"]
    .mean()
    .sort_values(ascending=False)
)

print("Promedio general según tipo de almuerzo:")
print(promedio_lunch.round(2))


In [ ]:
# Tabla resumen de algunos grupos
resumen_grupos = pd.DataFrame({
    "Curso preparación": df_clean.groupby("test_preparation")["average_score"].mean(),
})

resumen_grupos.round(2)


## 12. Conclusiones

A partir del análisis exploratorio se pueden obtener las siguientes conclusiones:

1. Se analizaron **1,000 registros y 8 variables**. El dataset contiene información sobre características de los estudiantes y sus calificaciones en Matemáticas, Lectura y Escritura.

2. No se encontraron **valores faltantes ni registros duplicados**, por lo que no fue necesario eliminar información durante la limpieza.

3. La variable `average_score` permite resumir el desempeño de cada estudiante utilizando sus resultados en las tres áreas.

4. La clasificación utilizada divide el rendimiento en **Bajo, Medio y Alto**, utilizando como límites 60 y 80 puntos.

5. Los promedios de Matemáticas, Lectura y Escritura permiten comparar cuál área presenta el desempeño promedio más alto.

6. El análisis del curso de preparación permite observar si existen diferencias en el promedio general entre quienes completaron el curso y quienes no lo realizaron. Esta diferencia describe una relación dentro de este dataset, pero por sí sola no demuestra que el curso sea la causa de los resultados.

7. El nivel educativo de los padres también presenta diferencias entre grupos en cuanto al promedio general. Estas diferencias son descriptivas y no implican necesariamente una relación causal.

8. Finalmente, la distribución de `average_score` permite identificar qué proporción de estudiantes se encuentra en los niveles Bajo, Medio y Alto y observar cómo se distribuyen los resultados académicos.


## 13. Dataset final

Al finalizar el procesamiento, el dataset contiene las variables originales y las nuevas variables `average_score` y `performance`.


In [ ]:
print("Dimensiones finales:", df_clean.shape)
print("\nColumnas finales:")
print(df_clean.columns.tolist())

df_clean.head()
